# Nonlinear Fourier retraction

This tutorial shows how to apply retraction using only public `qsppack` functions. Retraction maps a definite-parity Chebyshev polynomial to a QSP-feasible polynomial through Weiss spectral factorization and the inverse and forward nonlinear Fourier transforms.

In [ ]:
%matplotlib inline

from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np

from qsppack import cvx_poly_coef, retract

## Construct a polynomial to retract

As a representative example, approximate uniform singular-value amplification, $g_{\mathrm{SV}}(x)=x/a$, on $[0,a]$ with $a=0.2$. An odd degree-101 fit with 128 fitting points provides the polynomial that we will retract. The fitted polynomial can exceed one between constraint samples; odd parity determines its behavior on the negative half of the QSP domain.

In [ ]:
degree = 101
a = 0.2
npts = 128
n_weiss = 2**18

def target(x):
    return np.asarray(x) / a

fit_options = {
    "intervals": [0.0, a],
    "npts": npts,
    "objnorm": np.inf,
    "epsil": 0.0,
    "fscale": 1.0,
    "method": "cvxpy",
    "solver": "CLARABEL",
}
polynomial = cvx_poly_coef(target, degree, fit_options)

## Apply retraction

The high-level call accepts full ascending Chebyshev coefficients. It infers odd parity, runs the NLFA pipeline, and returns both the original and retracted coefficients with audit data.

In [ ]:
start = perf_counter()
result = retract(polynomial, n_weiss=n_weiss)
retraction_runtime = perf_counter() - start

print(f"degree={result.degree}, parity={result.parity}, N_weiss={result.n_weiss}")
print(f"original max |P|:  {result.original_metrics.max_magnitude:.12f}")
print(f"retracted max |P|: {result.metrics.max_magnitude:.12f}")
print(f"constraint violation: {result.metrics.max_constraint_violation:.3e}")
print(f"NLFA reconstruction residual: {result.reconstruction_residual:.3e}")
print(f"retraction runtime: {retraction_runtime:.3f} s")

### How feasibility is checked

A uniform plot grid is not used as the certificate. For a real polynomial, the global extrema on $[-1,1]$ occur at the two endpoints or at real roots of its derivative. `RetractionMetrics` evaluates all of those candidates and reports the largest magnitude and its location.

In [ ]:
max_point = result.metrics.maximizer
print(f"checked {len(result.metrics.critical_points)} endpoint/critical-point candidates")
print(f"global maximizer x={max_point:.12f}")
assert result.metrics.max_constraint_violation < 1e-10

## Compare the original and retracted polynomials

The left panel compares the target, original polynomial, and retraction on $[0,1]$. The right panel compares pointwise errors on the fitted interval $[0,0.2]$. The scaled baseline divides the original polynomial by its true critical-point maximum.

In [ ]:
x = np.linspace(0.0, 1.0, 5000)
x_fit = np.linspace(0.0, a, 2000)
p = result.evaluate(x, "original")
p_retracted = result.evaluate(x)
scale = result.original_metrics.max_magnitude
floor = 1e-16  # Keep exact zeros visible without flattening the log-scale plot.

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4.5))
left.plot(x, target(x), color="black", label=r"$g_{\mathrm{SV}}$")
left.plot(x, p, color="#0072B2", label=r"$P$")
left.plot(x, p_retracted, "--", color="#E69F00", label=r"$\mathcal{R}(P)$")
left.set(xlim=(0, 1), ylim=(-1.1, 1.1), xlabel=r"$x$")
left.grid(alpha=0.3)
left.legend(loc="lower right")

target_fit = target(x_fit)
error_retracted = np.maximum(np.abs(result.evaluate(x_fit) - target_fit), floor)
error_scaled = np.maximum(np.abs(result.evaluate(x_fit, "original") / scale - target_fit), floor)
error_original = np.maximum(np.abs(result.evaluate(x_fit, "original") - target_fit), floor)
right.plot(x_fit, error_scaled, color="#009E73", label=r"Scaled $P$")
right.plot(x_fit, error_original, color="#0072B2", label=r"$P$")
right.plot(x_fit, error_retracted, "--", color="#E69F00", label=r"$\mathcal{R}(P)$")
right.set(xlim=(0, a), xlabel=r"$x$", ylabel=r"$|P(x)-g_{\mathrm{SV}}(x)|$", yscale="log")
right.grid(alpha=0.3, which="both")
right.legend(loc="lower right")
fig.tight_layout()
plt.show()

## Weiss resolution

`N_weiss` controls the roots-of-unity discretization used for spectral factorization. The comparison below reuses exactly the same unretracted degree-101 polynomial, so any change is caused by retraction resolution alone. The final row uses the high-resolution setting from the main example.

In [ ]:
comparisons = []
for resolution in (2**12, 2**14):
    start = perf_counter()
    candidate = retract(polynomial, n_weiss=resolution)
    elapsed = perf_counter() - start
    comparisons.append((resolution, elapsed, candidate))
comparisons.append((n_weiss, retraction_runtime, result))

print(f"{'N_weiss':>10} {'runtime (s)':>12} {'max |P|':>14} {'fit error':>14} {'NLFA residual':>15}")
for resolution, elapsed, candidate in comparisons:
    fit_error = np.max(np.abs(candidate.evaluate(x_fit) - target_fit))
    print(f"{resolution:10d} {elapsed:12.3f} {candidate.metrics.max_magnitude:14.12f} {fit_error:14.6e} {candidate.reconstruction_residual:15.6e}")

Retraction enforces the QSP magnitude constraint while preserving polynomial parity and degree. Its finite-`N_weiss` result need not equal simple uniform rescaling, so approximation error and reconstruction residual should be reported together with the global feasibility metric.